In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("catalogo", "catalog_ecobici_dev")
catalogo = dbutils.widgets.get("catalogo")

In [0]:
tabla_bronze     = f"{catalogo}.bronze.viajes_raw"
tabla_estaciones = f"{catalogo}.silver.estaciones"
tabla_silver     = f"{catalogo}.silver.viajes"
tabla_cuarentena = f"{catalogo}.silver.viajes_cuarentena"
tabla_resumen    = f"{catalogo}.silver.resumen_calidad_viajes"

In [0]:
##Parámetros de las reglas
EDAD_MIN, EDAD_MAX = 14, 90
DURACION_MAX_MIN   = 1440          # 24 horas
GENEROS_VALIDOS    = ["M", "F", "O"]

In [0]:
bronze = spark.table(tabla_bronze)

In [0]:
def a_entero(columna, tipo="int", digitos=9):
    """Texto -> número entero, o NULL si no tiene forma de entero (acepta ceros a la izquierda: '028' -> 28)."""
    texto = F.trim(columna)
    return F.when(texto.rlike(rf"^\d{{1,{digitos}}}$"), texto.cast(tipo))

In [0]:
estacion_retiro = F.split(F.trim("Ciclo_Estacion_Retiro"), "-")[0]
estacion_arribo = F.split(F.trim("Ciclo_EstacionArribo"), "-")[0]

In [0]:
edad_txt = F.trim("Edad_Usuario")

In [0]:
# El archivo se llama ecobici_AAAA_MM.csv: de ahí se tomara el periodo (AAAAMM) al que pertenece
patron_archivo = r"ecobici_(\d{4})_(\d{2})"
anio_archivo = F.regexp_extract("_archivo_origen", patron_archivo, 1)
mes_archivo  = F.regexp_extract("_archivo_origen", patron_archivo, 2)

In [0]:
tipado = (
    bronze
    .select(
        "*",
        # M, F u O se conservan; cualquier otro valor ("?", vacío, nulo) pasa a "ND" (no disponible)
        F.when(F.upper(F.trim("Genero_Usuario")).isin(GENEROS_VALIDOS), F.upper(F.trim("Genero_Usuario")))
         .otherwise(F.lit("ND")).alias("genero"),
        F.when(edad_txt.rlike(r"^\d{1,9}(\.\d+)?$"), edad_txt.cast("double").cast("int")).alias("edad"),
        a_entero("Bici", "bigint", 18).alias("bici_id"),
        # "271-272" -> 271 (primer número); "028" -> 28; texto que no es número -> NULL
        a_entero(estacion_retiro).alias("estacion_retiro"),
        a_entero(estacion_arribo).alias("estacion_arribo"),
        F.coalesce(F.col("Ciclo_Estacion_Retiro").contains("-"), F.lit(False)).alias("estacion_retiro_es_rango"),
        F.coalesce(F.col("Ciclo_EstacionArribo").contains("-"), F.lit(False)).alias("estacion_arribo_es_rango"),
        F.try_to_timestamp(
            F.concat_ws(" ", "Fecha_Retiro", "Hora_Retiro"), F.lit("dd/MM/yyyy HH:mm:ss")
        ).alias("ts_retiro"),
        F.try_to_timestamp(
            F.concat_ws(" ", "Fecha_Arribo", "Hora_Arribo"), F.lit("dd/MM/yyyy HH:mm:ss")
        ).alias("ts_arribo"),
        F.when(anio_archivo != "", F.concat(anio_archivo, mes_archivo).cast("int"))
         .alias("periodo_archivo"),
    )
    .withColumn(
        "duracion_min",
        (F.col("ts_arribo").cast("long") - F.col("ts_retiro").cast("long")) / 60,
    )
    .withColumn("periodo_retiro", F.date_format("ts_retiro", "yyyyMM").cast("int"))
    .withColumn("periodo_arribo", F.date_format("ts_arribo", "yyyyMM").cast("int"))
)

In [0]:
ids_estaciones = [
    r["num_cicloe"] for r in spark.table(tabla_estaciones).select("num_cicloe").distinct().collect()
]
print("Estaciones en el catálogo:", len(ids_estaciones))
assert len(ids_estaciones) > 0, "El catálogo silver.estaciones está vacío: ejecuta primero 03_silver_estaciones"

Estaciones en el catálogo: 689


In [0]:
##LLave de negocio
llave = ["bici_id", "ts_retiro", "estacion_retiro", "ts_arribo", "estacion_arribo"]
w = Window.partitionBy(*llave).orderBy("_fecha_ingesta", "_archivo_origen")

In [0]:
clasificado = (
    tipado
    .withColumn("rn", F.row_number().over(w))
    .withColumn(
        "motivo_rechazo",
        F.when(F.col("ts_retiro").isNull() | F.col("ts_arribo").isNull(), "fecha_invalida")
         .when(F.col("estacion_retiro").isNull() | F.col("estacion_arribo").isNull(), "estacion_invalida")
         .when(
             F.col("edad").isNull() | (F.col("edad") < EDAD_MIN) | (F.col("edad") > EDAD_MAX),
             "edad_fuera_de_rango",
         )
         .when(F.col("ts_arribo") < F.col("ts_retiro"), "arribo_antes_de_retiro")
         .when(F.col("duracion_min") == 0, "duracion_cero")
         .when(F.col("duracion_min") > DURACION_MAX_MIN, "duracion_excesiva")
         .when(
             (F.col("periodo_retiro") != F.col("periodo_archivo"))
             & (F.col("periodo_arribo") != F.col("periodo_archivo")),
             "fuera_del_mes_del_archivo",
         )
         .when(
             ~F.col("estacion_retiro").isin(ids_estaciones)
             | ~F.col("estacion_arribo").isin(ids_estaciones),
             "estacion_no_en_catalogo",
         )
         .when(F.col("rn") > 1, "duplicado")
    )
)

In [0]:
validos = (
    clasificado
    .filter(F.col("motivo_rechazo").isNull())
    .select(
        F.sha2(
            F.concat_ws("|", "bici_id", "ts_retiro", "estacion_retiro", "ts_arribo", "estacion_arribo"), 256
        ).alias("viaje_id"),
        "genero",
        "edad",
        "bici_id",
        "estacion_retiro",
        "estacion_arribo",
        "estacion_retiro_es_rango",
        "estacion_arribo_es_rango",
        "ts_retiro",
        "ts_arribo",
        "duracion_min",
        F.to_date("ts_retiro").alias("fecha_retiro"),
        F.date_trunc("hour", "ts_retiro").alias("hora_clima"),   # llave para cruzar con el clima por hora
        "periodo_archivo",
        "_archivo_origen",
        "_fecha_ingesta",
    )
)

In [0]:
cuarentena = clasificado.filter(F.col("motivo_rechazo").isNotNull()).drop("rn")


In [0]:
(
    validos.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(tabla_silver)
)

In [0]:
(
    cuarentena.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(tabla_cuarentena)
)

In [0]:
total_bronze = spark.table(tabla_bronze).count()
n_validos    = spark.table(tabla_silver).count()
n_cuarentena = spark.table(tabla_cuarentena).count()

In [0]:
print(f"Bronce:     {total_bronze:,}")
print(f"Silver:     {n_validos:,}")
print(f"Cuarentena: {n_cuarentena:,}")

Bronce:     6,065,485
Silver:     6,065,074
Cuarentena: 411


In [0]:
assert total_bronze == n_validos + n_cuarentena, "Se perdieron o duplicaron filas"

In [0]:
por_motivo = (
    spark.table(tabla_cuarentena)
    .groupBy("motivo_rechazo")
    .agg(F.count("*").alias("registros"))
    .withColumnRenamed("motivo_rechazo", "resultado")
)

In [0]:
fila_validos = spark.createDataFrame([("validos", n_validos)], "resultado string, registros long")

In [0]:
resumen = (
    fila_validos.unionByName(por_motivo)
    .withColumn("porcentaje", F.round(F.col("registros") / F.lit(total_bronze) * 100, 3))
    .withColumn("ejecutado_en", F.current_timestamp())
    .orderBy(F.desc("registros"))
)

In [0]:
(
    resumen.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(tabla_resumen)
)
 
display(spark.table(tabla_resumen))

resultado,registros,porcentaje,ejecutado_en
validos,6065074,99.993,2026-11-14T01:34:41.475527Z
duracion_excesiva,180,0.003,2026-11-14T01:34:41.475527Z
edad_fuera_de_rango,155,0.003,2026-11-14T01:34:41.475527Z
estacion_invalida,44,0.001,2026-11-14T01:34:41.475527Z
estacion_no_en_catalogo,32,0.001,2026-11-14T01:34:41.475527Z


In [0]:
silver = spark.table(tabla_silver)

In [0]:
display(silver.select("edad", "duracion_min").summary("min", "1%", "25%", "50%", "75%", "99%", "max"))

summary,edad,duracion_min
min,16,0.21666666666666667
1%,19,2.25
25%,28,7.116666666666666
50%,33,11.883333333333333
75%,40,19.966666666666665
99%,64,52.28333333333333
max,90,1431.5666666666666


In [0]:
display(silver.groupBy("genero").count().orderBy(F.desc("count")))

genero,count
M,4121614
F,1678677
ND,158131
O,106652


In [0]:
display(
    silver.agg(
        F.count("*").alias("viajes"),
        F.sum(F.col("estacion_retiro_es_rango").cast("int")).alias("retiro_con_rango"),
        F.sum(F.col("estacion_arribo_es_rango").cast("int")).alias("arribo_con_rango"),
    )
)

viajes,retiro_con_rango,arribo_con_rango
6065074,231311,251850


In [0]:
display(spark.table(tabla_cuarentena).limit(20))

Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_EstacionArribo,Fecha_Arribo,Hora_Arribo,_rescued_data,_archivo_origen,_fecha_ingesta,genero,edad,bici_id,estacion_retiro,estacion_arribo,estacion_retiro_es_rango,estacion_arribo_es_rango,ts_retiro,ts_arribo,periodo_archivo,duracion_min,periodo_retiro,periodo_arribo,motivo_rechazo
F,28.0,2036523,237-238,20/08/2026,18:15:24,105,16/09/2026,17:40:31,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_09.csv,2026-11-12T21:18:19.812Z,F,28,2036523,237,105,true,false,2026-08-20T18:15:24Z,2026-09-16T17:40:31Z,202609,38845.11666666667,202608,202609,duracion_excesiva
F,31.0,2049815,688,01/07/2026,20:18:48,704,02/07/2026,21:13:29,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_07.csv,2026-11-12T21:18:19.812Z,F,31,2049815,688,704,false,false,2026-07-01T20:18:48Z,2026-07-02T21:13:29Z,202607,1494.6833333333334,202607,202607,duracion_excesiva
F,null,2386373,521,29/07/2026,19:40:50,525,29/07/2026,19:49:21,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_07.csv,2026-11-12T21:18:19.812Z,F,null,2386373,521,525,false,false,2026-07-29T19:40:50Z,2026-07-29T19:49:21Z,202607,8.516666666666667,202607,202607,edad_fuera_de_rango
M,45.0,2473353,475,02/08/2026,08:45:20,233,03/08/2026,09:35:24,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_08.csv,2026-11-12T21:18:19.812Z,M,45,2473353,475,233,false,false,2026-08-02T08:45:20Z,2026-08-03T09:35:24Z,202608,1490.0666666666666,202608,202608,duracion_excesiva
M,null,2806099,036,21/10/2026,17:46:10,472,21/10/2026,17:58:44,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_10.csv,2026-11-12T21:18:19.812Z,M,null,2806099,36,472,false,false,2026-10-21T17:46:10Z,2026-10-21T17:58:44Z,202610,12.566666666666666,202610,202610,edad_fuera_de_rango
M,21.0,2880375,065,19/07/2026,11:02:49,1000,19/07/2026,14:05:00,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_07.csv,2026-11-12T21:18:19.812Z,M,21,2880375,65,1000,false,false,2026-07-19T11:02:49Z,2026-07-19T14:05:00Z,202607,182.18333333333334,202607,202607,estacion_no_en_catalogo
M,30.0,2987238,043,05/07/2026,08:31:41,Temporal 2,05/07/2026,10:29:33,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_07.csv,2026-11-12T21:18:19.812Z,M,30,2987238,43,null,false,false,2026-07-05T08:31:41Z,2026-07-05T10:29:33Z,202607,117.86666666666666,202607,202607,estacion_invalida
F,28.0,3009974,114,21/07/2026,23:13:36,1000,29/07/2026,14:01:28,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_07.csv,2026-11-12T21:18:19.812Z,F,28,3009974,114,1000,false,false,2026-07-21T23:13:36Z,2026-07-29T14:01:28Z,202607,10967.866666666667,202607,202607,duracion_excesiva
M,27.0,3056892,072,11/07/2026,07:52:45,154,16/07/2026,09:05:35,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_07.csv,2026-11-12T21:18:19.812Z,M,27,3056892,72,154,false,false,2026-07-11T07:52:45Z,2026-07-16T09:05:35Z,202607,7272.833333333333,202607,202607,duracion_excesiva
M,null,3067022,351,23/10/2026,07:11:56,565,23/10/2026,07:43:16,null,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/viajes/ecobici_2026_10.csv,2026-11-12T21:18:19.812Z,M,null,3067022,351,565,false,false,2026-10-23T07:11:56Z,2026-10-23T07:43:16Z,202610,31.333333333333332,202610,202610,edad_fuera_de_rango
